# Crypto Forecasters — Evaluación individual por modelo (Colab)

Evalúa **cada modelo en su propia celda** con **walk-forward 80/20** (la prueba que sí cubre todo el 20% de test). Corre una celda, anota el resultado, sigue con la siguiente.

Modelos: **N-HiTS, LightGBM, GRU, Assembly, Chronos** (TFT fuera).

Protocolo único: **80/20 walk-forward** con horizonte de **7 días**.
- Split: 80% train / 20% test por posición de fila (cronológico, sin mezclar).
- Desliza ventanas de 7 días por todo el 20%, re-entrenando cada vez.
- Reporta **MAPE promedio ± std** y también **MAPE día 1 vs día 7** (para el paper).

**Para cambiar de cripto:** edita `EVAL_SYMBOL` en la celda 7 (Setup) y re-corre esa celda + las celdas de los modelos.

**Antes de correr:** Runtime → Change runtime type → T4 GPU.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > GPU')

## 2. Install dependencies

In [ ]:
%%capture
!pip install neuralforecast>=1.7.0 lightgbm>=4.0.0 yfinance joblib scikit-learn scipy chronos-forecasting

## 3. Mount Google Drive and clone/pull the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

REPO_DIR    = '/content/drive/MyDrive/capstone_project_unfc'
REPO_URL    = 'https://github.com/RocioT08/capstone_project_unfc.git'
BACKEND_DIR = os.path.join(REPO_DIR, 'backend')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)
print('Backend dir:', BACKEND_DIR)

## 4. Configuration (parámetros fijos para todos los tickers)

In [ ]:
import random
import numpy as np
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

TICKERS_WITH_FEAR_GREED = {
    'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD',
    'ADA-USD', 'AVAX-USD', 'DOGE-USD',
}

DATA_START       = '2016-01-01'
CONFIDENCE_LEVEL = 0.95
HORIZON          = 7            # 7 días = lo que se muestra en el front

# ---- walk-forward 80/20 ----------------------------------------------------
# WALK_STEP=14 → ~45 ventanas (cubre todo el 20% pero la mitad de tiempo que step=7).
# Sube a 7 para ~89 ventanas (más fino, más lento). TODOS los modelos usan el mismo
# valor para que la comparación sea justa.
WALK_STEP   = 14
MAX_WINDOWS = None             # None = cubre TODO el 20%. Pon un número para limitar.

print('Horizon   :', HORIZON, 'días')
print('Walk step :', WALK_STEP, 'días')

## 5. Data helpers (yfinance)

In [ ]:
import pandas as pd
import yfinance as yf
import math

def fetch_ohlcv_yf(symbol: str, start: str = DATA_START) -> pd.DataFrame:
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=start, auto_adjust=True)
    df.index = pd.to_datetime(df.index, utc=True)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)
    df = df.sort_index().dropna()
    print(f'{symbol}: {len(df)} rows from {df.index[0].date()} to {df.index[-1].date()}')
    return df

def _compute_error_metrics(actuals, predictions):
    mae  = float(np.mean([abs(a - p) for a, p in zip(actuals, predictions)]))
    rmse = float(math.sqrt(np.mean([(a - p)**2 for a, p in zip(actuals, predictions)])))
    mape = float(np.mean([abs(a - p) / abs(a) * 100 for a, p in zip(actuals, predictions) if a != 0]))
    return {'mae': round(mae, 4), 'rmse': round(rmse, 4), 'mape': round(mape, 4)}

## 6. Import models

In [ ]:
from analytics.forecasting.crypto.nhits_forecaster import NHiTSForecaster, _fetch_fear_greed
from analytics.forecasting.crypto.lightgbm_forecaster import LightGBMForecaster
from analytics.forecasting.crypto.gru import GRUForecaster
from analytics.forecasting.crypto.assembly import CryptoAssemblyForecaster
from analytics.forecasting import chronos2
print('Imports OK')

## 7. Setup walk-forward  👈 CAMBIA EL TICKER AQUÍ

Edita `EVAL_SYMBOL` abajo y re-corre **solo esta celda** + las celdas de los modelos. Carga los datos, hace el split 80/20 y define `run_walkforward()`. Los resultados se acumulan en `WF_RESULTS`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
EVAL_SYMBOL = 'BNB-USD'   # 👈 CAMBIA EL TICKER AQUÍ (BTC-USD, ETH-USD, SOL-USD, ...)
# ═══════════════════════════════════════════════════════════════════════════

# ── Datos + Fear & Greed ─────────────────────────────────────────────────────
ohlcv = fetch_ohlcv_yf(EVAL_SYMBOL)

fear_greed = None
if EVAL_SYMBOL in TICKERS_WITH_FEAR_GREED:
    try:
        fear_greed = _fetch_fear_greed(n_days=3000)
        print(f'Fear & Greed: {len(fear_greed)} rows')
    except Exception as e:
        print(f'Fear & Greed unavailable: {e}')

# ── Split 80/20 por posición de fila (cronológico) ───────────────────────────
split_idx = int(len(ohlcv) * 0.80)
test_20   = ohlcv.iloc[split_idx:]

# Mismas ventanas para TODOS los modelos (comparación justa)
window_steps = list(range(0, len(test_20) - (HORIZON - 1), WALK_STEP))
if MAX_WINDOWS is not None:
    window_steps = window_steps[:MAX_WINDOWS]

print(f'Ticker     : {EVAL_SYMBOL}')
print(f'Train (80%): {split_idx} rows')
print(f'Test  (20%): {len(test_20)} rows')
print(f'Ventanas   : {len(window_steps)}  (step={WALK_STEP} días, horizonte={HORIZON})')

# Resultados acumulados (clave = (ticker, modelo) para no mezclar entre criptos)
try:
    WF_RESULTS
except NameError:
    WF_RESULTS = {}


def _fit_forecast(name, factory, train_ohlcv, periods=HORIZON):
    """Fit one model on train_ohlcv → return its point_forecast list."""
    if name == 'Chronos':
        return chronos2.forecast(train_ohlcv['Close'], periods, CONFIDENCE_LEVEL, '1d')['point_forecast']
    model = factory()
    try:
        model.fit(train_ohlcv, fear_greed=fear_greed)   # N-HiTS / Assembly
    except TypeError:
        model.fit(train_ohlcv)                            # GRU / LightGBM
    return model.forecast(periods=periods)['point_forecast']


def run_walkforward(name, factory):
    """Run the 80/20 walk-forward for ONE model on EVAL_SYMBOL. Saves to WF_RESULTS."""
    rows, perday = [], []
    print(f"Walk-forward: {EVAL_SYMBOL} · {name}  ({len(window_steps)} ventanas)\n")
    for i, step in enumerate(window_steps, 1):
        ctx_end    = split_idx + step
        actual_end = ctx_end + HORIZON
        if actual_end > len(ohlcv):
            break
        context = ohlcv.iloc[:ctx_end]
        actuals = ohlcv['Close'].iloc[ctx_end:actual_end].values
        label   = str(ohlcv.index[ctx_end].date())
        try:
            preds = _fit_forecast(name, factory, context, HORIZON)
            m = _compute_error_metrics(actuals.tolist(), preds)
            rows.append({'window': label, **m})
            ape = [abs(a - p) / abs(a) * 100 for a, p in zip(actuals, preds) if a != 0]
            if len(ape) == HORIZON:
                perday.append(ape)
            print(f"  [{i:>2}/{len(window_steps)}] {label}  MAPE={m['mape']:.4f}%")
        except Exception as e:
            print(f"  [{i:>2}/{len(window_steps)}] {label}  ERROR — {e}")

    df = pd.DataFrame(rows)
    perday_arr = np.array(perday) if perday else np.zeros((1, HORIZON))
    summary = {
        'ticker':    EVAL_SYMBOL,
        'model':     name,
        'mape_mean': round(df['mape'].mean(), 4),
        'mape_std':  round(df['mape'].std(), 4),
        'mae_mean':  round(df['mae'].mean(), 4),
        'rmse_mean': round(df['rmse'].mean(), 4),
        'mape_day1': round(float(perday_arr[:, 0].mean()), 4),
        'mape_day7': round(float(perday_arr[:, -1].mean()), 4),
        'n_windows': len(df),
    }
    WF_RESULTS[(EVAL_SYMBOL, name)] = summary
    print(f"\n{'='*56}")
    print(f"  {EVAL_SYMBOL} · {name}")
    print(f"  MAPE promedio : {summary['mape_mean']:.4f}%  ±{summary['mape_std']:.4f}")
    print(f"  MAE  promedio : {summary['mae_mean']:.4f}")
    print(f"  RMSE promedio : {summary['rmse_mean']:.4f}")
    print(f"  MAPE día 1    : {summary['mape_day1']:.4f}%   (más fácil)")
    print(f"  MAPE día 7    : {summary['mape_day7']:.4f}%   (más difícil)")
    print(f"  Ventanas      : {summary['n_windows']}")
    print(f"{'='*56}")
    return df

print('\nSetup listo para', EVAL_SYMBOL, '— ahora corre cada celda de modelo.')

## 8. N-HiTS  ← corre y anota el resultado

In [ ]:
df_nhits = run_walkforward(
    'N-HiTS',
    lambda: NHiTSForecaster(max_horizon=HORIZON, confidence_level=CONFIDENCE_LEVEL,
                            max_steps=500, input_size=120),
)

## 9. LightGBM  ← corre y anota el resultado

In [ ]:
df_lgb = run_walkforward(
    'LightGBM',
    lambda: LightGBMForecaster(max_horizon=HORIZON, confidence_level=CONFIDENCE_LEVEL,
                               lags=28, n_estimators=300, learning_rate=0.03, num_leaves=63),
)

## 10. GRU  ← corre y anota el resultado

In [ ]:
df_gru = run_walkforward(
    'GRU',
    lambda: GRUForecaster(max_horizon=HORIZON, confidence_level=CONFIDENCE_LEVEL,
                          epochs=20, mc_samples=40, lookback=60, hidden_size=128, num_layers=3),
)

## 11. Chronos (benchmark zero-shot)  ← corre y anota el resultado

In [ ]:
df_chronos = run_walkforward('Chronos', None)   # Chronos no necesita factory

## 12. Assembly (ensemble) ⚠️ LENTO — córrelo al final / de noche

Re-entrena GRU + N-HiTS + LightGBM en cada ventana, así que toma mucho más tiempo.
Usa las mismas ventanas que los demás para que la comparación sea justa.

In [ ]:
df_assembly = run_walkforward(
    'Assembly',
    lambda: CryptoAssemblyForecaster(
        max_horizon=HORIZON, n_splits=4, ridge_alpha=0.5, min_train_size=120,
        confidence_level=CONFIDENCE_LEVEL, use_gru=True, use_tft=False,
        gru_kwargs={'epochs': 20, 'mc_samples': 40, 'lookback': 60, 'hidden_size': 128, 'num_layers': 3},
        nhits_kwargs={'max_steps': 500, 'input_size': 120},
        lgb_kwargs={'lags': 28, 'n_estimators': 300, 'learning_rate': 0.03, 'num_leaves': 63},
    ),
)

## 13. Comparación final — modelos del ticker actual

In [ ]:
rows = [v for (tk, _), v in WF_RESULTS.items() if tk == EVAL_SYMBOL]
if not rows:
    print('Corre al menos una celda de modelo primero.')
else:
    final = pd.DataFrame(rows).set_index('model')
    final = final[['mape_mean', 'mape_std', 'mae_mean', 'rmse_mean',
                   'mape_day1', 'mape_day7', 'n_windows']].sort_values('mape_mean')
    print('=' * 70)
    print(f'  RANKING FINAL — {EVAL_SYMBOL}  (menor MAPE = mejor)')
    print('=' * 70)
    print(final.to_string())
    print('=' * 70)
    print('Mejor modelo:', final.index[0])

In [ ]:
# Gráfico: MAPE por modelo (± std) y día 1 vs día 7 — ticker actual
import matplotlib.pyplot as plt

rows = [v for (tk, _), v in WF_RESULTS.items() if tk == EVAL_SYMBOL]
if rows:
    final = pd.DataFrame(rows).set_index('model').sort_values('mape_mean')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].bar(final.index, final['mape_mean'], yerr=final['mape_std'], capsize=5, color='steelblue')
    axes[0].set_ylabel('MAPE promedio (%)'); axes[0].set_title(f'{EVAL_SYMBOL} — precisión por modelo (± std)')

    x = np.arange(len(final)); w = 0.35
    axes[1].bar(x - w/2, final['mape_day1'], w, label='Día 1')
    axes[1].bar(x + w/2, final['mape_day7'], w, label='Día 7')
    axes[1].set_xticks(x); axes[1].set_xticklabels(final.index)
    axes[1].set_ylabel('MAPE (%)'); axes[1].set_title('Día 1 vs Día 7 (cómo crece el error)'); axes[1].legend()

    plt.tight_layout(); plt.show()

In [ ]:
# Guardar TODOS los resultados (todos los tickers que hayas corrido) a Drive
import os
OUT_DIR = '/content/drive/MyDrive/capstone_checkpoints'
os.makedirs(OUT_DIR, exist_ok=True)
if WF_RESULTS:
    pd.DataFrame(list(WF_RESULTS.values())).to_csv(
        os.path.join(OUT_DIR, 'walkforward_ranking_ALL.csv'), index=False)
    print('Guardado:', os.path.join(OUT_DIR, 'walkforward_ranking_ALL.csv'))
    print('Tickers guardados:', sorted({tk for tk, _ in WF_RESULTS}))